<a href="https://colab.research.google.com/github/FardinLakdawala/prml-assigments/blob/main/prml_3_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Iris Dataset Classification & Resampling Analysis

This notebook covers:
1. Linear Regression Classifiers (One-vs-Rest) with SGD on 80:10:10 and 70:15:15 splits.
2. Oversampling to balance the dataset and evaluation.
3. SMOTE implementation from scratch with two custom settings:
   - **Setting A**: Generate synthetic points from any 2 random samples of the minority class.
   - **Setting B**: Generate synthetic points using the nearest neighbor approach.
4. Custom Logistic Regression with Stochastic Gradient Descent from scratch.

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

# Convert to DataFrame for easier inspection
df = pd.DataFrame(X, columns=iris.feature_names)
df['species'] = y
display(df.head())

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## Helper Functions for Splits and Custom SGD Linear Regression Classifier

Since Linear Regression outputs continuous values, we build a binary classifier for each category (One-vs-Rest) using a threshold of `0.5` on the regression output.

In [8]:
def train_val_test_split(X, y, val_ratio, test_ratio, random_state=42):
    np.random.seed(random_state)
    perm = np.random.permutation(len(X))
    X_shuffled, y_shuffled = X[perm], y[perm]

    test_bound = int(len(X) * (1 - test_ratio))
    val_bound = int(len(X) * (1 - test_ratio - val_ratio))

    X_train, y_train = X_shuffled[:val_bound], y_shuffled[:val_bound]
    X_val, y_val = X_shuffled[val_bound:test_bound], y_shuffled[val_bound:test_bound]
    X_test, y_test = X_shuffled[test_bound:], y_shuffled[test_bound:]

    return X_train, X_val, X_test, y_train, y_val, y_test

class SGDLinearRegressionClassifier:
    def __init__(self, lr=0.01, epochs=100):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)
            for idx in indices:
                xi = X[idx]
                yi = y[idx]
                prediction = np.dot(xi, self.weights) + self.bias
                error = prediction - yi

                # Gradient steps
                self.weights -= self.lr * error * xi
                self.bias -= self.lr * error

    def predict_proba(self, X):
        return np.dot(X, self.weights) + self.bias

def train_one_vs_rest(X_train, y_train, X_val, X_test, lr=0.01, epochs=150):
    classifiers = {}
    val_preds = {}
    test_preds = {}

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    for class_idx in np.unique(y_train):
        # Binary target for One-vs-Rest
        y_train_binary = (y_train == class_idx).astype(int)

        clf = SGDLinearRegressionClassifier(lr=lr, epochs=epochs)
        clf.fit(X_train_scaled, y_train_binary)
        classifiers[class_idx] = clf

        val_preds[class_idx] = clf.predict_proba(X_val_scaled)
        test_preds[class_idx] = clf.predict_proba(X_test_scaled)

    # Argmax strategy to assign final class label
    val_final_pred = np.argmax(np.column_stack([val_preds[c] for c in sorted(classifiers.keys())]), axis=1)
    test_final_pred = np.argmax(np.column_stack([test_preds[c] for c in sorted(classifiers.keys())]), axis=1)

    return val_final_pred, test_final_pred

## Part 1: Linear Regression Classifier on Standard Splits

In [9]:
for split_name, val_r, test_r in [('80:10:10', 0.1, 0.1), ('70:15:15', 0.15, 0.15)]:
    X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y, val_r, test_r)
    val_pred, test_pred = train_one_vs_rest(X_train, y_train, X_val, X_test)

    print(f"=== Performance for Split Proportions {split_name} ===")
    print(f"Validation Accuracy: {accuracy_score(y_val, val_pred):.4f}")
    print(f"Test Accuracy: {accuracy_score(y_test, test_pred):.4f}\n")

=== Performance for Split Proportions 80:10:10 ===
Validation Accuracy: 0.8667
Test Accuracy: 0.8000

=== Performance for Split Proportions 70:15:15 ===
Validation Accuracy: 0.8636
Test Accuracy: 0.8696



## Part 2: Oversampling Techniques to Balance Classes

In [10]:
def random_oversample(X_train, y_train):
    classes, counts = np.unique(y_train, return_counts=True)
    max_count = max(counts)

    X_resampled, y_resampled = list(X_train), list(y_train)

    for cls in classes:
        cls_indices = np.where(y_train == cls)[0]
        num_to_add = max_count - len(cls_indices)
        if num_to_add > 0:
            added_indices = np.random.choice(cls_indices, size=num_to_add, replace=True)
            for idx in added_indices:
                X_resampled.append(X_train[idx])
                y_resampled.append(y_train[idx])

    return np.array(X_resampled), np.array(y_resampled)

# Evaluate with Random Oversampling
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y, 0.1, 0.1)
X_train_over, y_train_over = random_oversample(X_train, y_train)
val_pred_over, test_pred_over = train_one_vs_rest(X_train_over, y_train_over, X_val, X_test)
print(f"Oversampled Train Target Count: {np.bincount(y_train_over)}")
print(f"Oversampled Test Accuracy: {accuracy_score(y_test, test_pred_over):.4f}")

Oversampled Train Target Count: [43 43 43]
Oversampled Test Accuracy: 0.8667


## Part 3: Custom SMOTE Implementation (Setting A & Setting B)

In [11]:
def custom_smote(X_train, y_train, setting='B', k_neighbors=3):
    classes, counts = np.unique(y_train, return_counts=True)
    max_count = max(counts)
    X_resampled, y_resampled = list(X_train), list(y_train)

    for cls in classes:
        cls_indices = np.where(y_train == cls)[0]
        cls_samples = X_train[cls_indices]
        num_to_add = max_count - len(cls_indices)

        if num_to_add <= 0:
            continue

        synthetic_samples = []
        for _ in range(num_to_add):
            if setting == 'A':
                # Setting A: Take any 2 random samples from the minority class
                idx1, idx2 = np.random.choice(len(cls_samples), size=2, replace=False)
                s1, s2 = cls_samples[idx1], cls_samples[idx2]
                diff = s2 - s1
                gap = np.random.rand()
                synthetic_samples.append(s1 + gap * diff)
            else:
                # Setting B: Find nearest neighbor in the class
                target_idx = np.random.choice(len(cls_samples))
                target_sample = cls_samples[target_idx]

                # Compute distances to all other samples in the same class
                dists = np.linalg.norm(cls_samples - target_sample, axis=1)
                sorted_indices = np.argsort(dists)

                # Select random index among nearest neighbors (excluding self)
                neighbor_idx = sorted_indices[np.random.choice(range(1, min(k_neighbors + 1, len(cls_samples))))]
                neighbor_sample = cls_samples[neighbor_idx]

                diff = neighbor_sample - target_sample
                gap = np.random.rand()
                synthetic_samples.append(target_sample + gap * diff)

        X_resampled.extend(synthetic_samples)
        y_resampled.extend([cls] * num_to_add)

    return np.array(X_resampled), np.array(y_resampled)

# Run Experiments for both Settings
for setting in ['A', 'B']:
    X_train_smote, y_train_smote = custom_smote(X_train, y_train, setting=setting)
    v_pred, t_pred = train_one_vs_rest(X_train_smote, y_train_smote, X_val, X_test)
    print(f"SMOTE Setting {setting} Test Accuracy: {accuracy_score(y_test, t_pred):.4f}")

SMOTE Setting A Test Accuracy: 0.8000
SMOTE Setting B Test Accuracy: 0.8000


## Part 4: Custom SGD Logistic Regression Classifier

In [12]:
class SGDLogisticRegressionClassifier:
    def __init__(self, lr=0.1, epochs=150):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)
            for idx in indices:
                xi = X[idx]
                yi = y[idx]
                prediction = self._sigmoid(np.dot(xi, self.weights) + self.bias)
                error = prediction - yi

                # Gradient descent step
                self.weights -= self.lr * error * xi
                self.bias -= self.lr * error

    def predict_proba(self, X):
        return self._sigmoid(np.dot(X, self.weights) + self.bias)

# Train Logistic Regression on One-vs-Rest configuration
def train_lr_one_vs_rest(X_train, y_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    classifiers = {}
    test_preds = {}

    for class_idx in np.unique(y_train):
        y_train_binary = (y_train == class_idx).astype(int)
        clf = SGDLogisticRegressionClassifier(lr=0.05, epochs=200)
        clf.fit(X_train_scaled, y_train_binary)
        classifiers[class_idx] = clf
        test_preds[class_idx] = clf.predict_proba(X_test_scaled)

    test_final_pred = np.argmax(np.column_stack([test_preds[c] for c in sorted(classifiers.keys())]), axis=1)
    return test_final_pred

# Evaluate custom logistic regression
lr_preds = train_lr_one_vs_rest(X_train, y_train, X_test)
print(f"Custom Logistic Regression Test Accuracy: {accuracy_score(y_test, lr_preds):.4f}")

Custom Logistic Regression Test Accuracy: 1.0000
